# GoTrain launch notebook

Run cells top to bottom from the repository root (this notebook lives in `notebooks/`, the first cell moves to the root).
Long jobs (baseline, agent) are started **in the background with nohup** so they survive browser disconnects; the notebook only launches and inspects them.
Everything here uses the train/validation splits only, except the clearly marked final-evaluation cell.

Terminal equivalents are in README.md. `%pip` lines are notebook-only; in a terminal use `python -m pip`.

In [ ]:
import os, sys, subprocess
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
PY = sys.executable
print("repo:", ROOT, "| python:", PY)

def go(*args, timeout=None):
    cmd = [PY, "-m", "gotrain", *args]
    print("$", " ".join(cmd))
    return subprocess.run(cmd, text=True, timeout=timeout)

def bg(name, *args):
    cmd = ["bash", "scripts/run_background.sh", name, *args]
    print("$", " ".join(cmd))
    return subprocess.run(cmd, text=True)

## 1. Environment check
Expected: torch 2.8.0 with `cuda_available: true`, `strands_import: ok`, `geobench_import: ok`, and `datasets_present` showing which tasks are already downloaded.
If a data library is missing (and only then):
```
%pip install "numpy==1.26.4" "rasterio==1.4.3" "geobench==1.1.0" "huggingface_hub==0.36.0"
```
then restart the kernel. Do not reinstall torch.

In [ ]:
go("env-check")

## 2. CPU test suite (a few minutes, no GPU needed)

In [ ]:
go("test")

## 3. Data: selective download, inspection, normalization
EuroSAT already exists at `~/dataset/geobench/classification_v1.0/m-eurosat` and is reused. The other three required tasks are fetched one zip at a time with SHA-256 verification.

In [ ]:
go("download", "--task", "all")

In [ ]:
go("inspect", "--task", "all", "--save", "--fit-normalization")

## 4. Smoke run: measures real speed on this GPU (short, partial data, labelled screening)
Use the printed `wall_minutes` and the per-epoch seconds in the experiment log to set `configs/budget.json` before the real runs.

In [ ]:
go("smoke", "--task", "all")

## 5. Fixed baselines for all four tasks (background)
Follow progress with `scripts/tail_logs.sh baseline` in a terminal, or the cell below.

In [ ]:
bg("baseline", "baseline", "--task", "all")

In [ ]:
subprocess.run(["bash", "scripts/status.sh"])

## 6. Autonomous agent (Sonnet on Bedrock), background
Adjust `--session-minutes` to the time you can give it. The agent only starts a run that fits the remaining budget; hard limits live in `configs/limits.json`.

In [ ]:
bg("agent", "agent", "--task", "all", "--session-minutes", "90")

In [ ]:
go("sessions")
go("list")

## 7. Dashboard (self-contained HTML; open it from the file browser or display inline)

In [ ]:
go("report")
from IPython.display import HTML, display
html = Path("gotrain_runs/reports/dashboard.html").read_text()
display(HTML(html))

## 8. Model registry: store, verify, promote (optional; freeze does this automatically)

In [ ]:
go("list-versions")
go("verify-model", "all")

## 9. Freeze the selection (no test data involved)

In [ ]:
go("freeze", "--task", "all")

## 10. FINAL TEST EVALUATION: run exactly once, after the freeze, at the end
This is the only command that touches the test split. Completed tasks are never re-evaluated.

In [ ]:
# go("final-eval", "--i-understand-this-uses-the-test-split")
print("uncomment the line above when the selection is frozen and you are ready")

In [ ]:
go("report")